# ROI Preprocessing

Refresh or normalize the canonical processed ROI dataset.

In [4]:
from __future__ import annotations

from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_02_ROI_PREPROCESSING_CELL_PROGRESS_1 = start_notebook_cell_progress('02_roi_preprocessing.ipynb', 'Load shared setup', total_steps=1)

import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from meatlens_pork_pipeline.image_ops import process_image

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())

finish_notebook_cell_progress(NB_02_ROI_PREPROCESSING_CELL_PROGRESS_1)


[START] 02_roi_preprocessing.ipynb | Load shared setup [0/1 step] elapsed=0.0s
[START] 00_shared_setup.ipynb | Shared setup bootstrap [0/1 step] elapsed=0.0s
Default processed dataset not ready at C:\Users\Adriaan M. Dimate\Desktop\development\school\MeatLens\meatlens-training-2\data\roboflow_processed_hsv_lab_threshold_roi_224 - run the early notebooks with raw or Excel input, or set overrides.
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [0/1 step] elapsed=0.0s
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [1/1 step] elapsed=0.0s
[RUNNING] 02_roi_preprocessing.ipynb | Load shared setup | done [0/1 step] elapsed=0.0s
[RUNNING] 02_roi_preprocessing.ipynb | Load shared setup | done [1/1 step] elapsed=0.0s


In [5]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_02_ROI_PREPROCESSING_CELL_PROGRESS_2 = start_notebook_cell_progress('02_roi_preprocessing.ipynb', 'Define preprocessing helpers', total_steps=1)

def preprocess_roi_image(path: Path, background_mode: str = 'gray') -> tuple[np.ndarray, dict[str, object]]:
    return process_image(path, background_mode=background_mode)


def preprocess_raw_center_crop_image(path: Path) -> tuple[np.ndarray, dict[str, object]]:
    image = Image.open(path).convert('RGB')
    if image.size != TARGET_SIZE:
        image = image.resize(TARGET_SIZE, Image.BILINEAR)
    return np.asarray(image, dtype=np.uint8), {
        'segmentation_failed': False,
        'mask_area_ratio': '',
        'center_overlap_ratio': '',
        'number_of_components': '',
        'touches_border': '',
    }


def build_processed_output_path(row: pd.Series, output_root: Path) -> Path:
    sample_number = str(row.get('sample_number', '')).strip()
    label = str(row['label']).strip()
    image_file_name = str(row['image_file_name']).strip()
    if sample_number:
        return output_root / f'sample {sample_number}' / label / image_file_name
    return output_root / label / image_file_name


def normalize_existing_processed_manifest(audited_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rows: list[dict[str, object]] = []
    summary_rows: list[dict[str, object]] = []
    for row in iter_notebook_progress(
        audited_df.to_dict(orient='records'),
        '02_roi_preprocessing.ipynb | normalize existing processed manifest',
        total=len(audited_df),
        unit='image',
        leave=True,
    ):
        local_path = Path(str(row['local_image_path']))
        record = dict(row)
        record['local_image_path'] = str(local_path)
        record['processed_output_file'] = str(local_path)
        record['input_mode'] = 'processed_hsv_lab_threshold_roi_224'
        rows.append(record)
        summary_rows.append(
            {
                'image_file_name': row['image_file_name'],
                'sample_number': row.get('sample_number', ''),
                'sample_id': row.get('sample_id', ''),
                'label': row['label'],
                'input_mode': 'processed_hsv_lab_threshold_roi_224',
                'processed_output_file': str(local_path),
                'segmentation_failed': str(row.get('segmentation_failed', 'False')),
                'mask_area_ratio': row.get('mask_area_ratio', ''),
                'center_overlap_ratio': row.get('center_overlap_ratio', ''),
                'number_of_components': row.get('number_of_components', ''),
                'touches_border': row.get('touches_border', ''),
            }
        )
    return pd.DataFrame(rows), pd.DataFrame(summary_rows), pd.DataFrame(columns=['image_file_name', 'error'])


def preprocess_audited_manifest(
    audited_df: pd.DataFrame,
    output_root: Path,
    input_mode: str,
    background_mode: str = 'gray',
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    records: list[dict[str, object]] = []
    summary_rows: list[dict[str, object]] = []
    failure_rows: list[dict[str, object]] = []

    for row in iter_notebook_progress(
        audited_df.to_dict(orient='records'),
        f'02_roi_preprocessing.ipynb | preprocess {input_mode}',
        total=len(audited_df),
        unit='image',
        leave=True,
    ):
        source_path = Path(str(row['local_image_path']))
        output_path = build_processed_output_path(pd.Series(row), output_root)
        ensure_dir(output_path.parent)
        try:
            if input_mode == 'processed_hsv_lab_threshold_roi_224':
                output_uint8, metadata = preprocess_roi_image(source_path, background_mode=background_mode)
            elif input_mode == 'raw_center_crop_224':
                output_uint8, metadata = preprocess_raw_center_crop_image(source_path)
            else:
                raise ValueError(f'Unsupported input_mode: {input_mode}')
            Image.fromarray(output_uint8).save(output_path)
            processed_record = dict(row)
            processed_record['local_image_path'] = str(output_path)
            processed_record['processed_output_file'] = str(output_path)
            processed_record['input_mode'] = input_mode
            processed_record.update(metadata)
            records.append(processed_record)
            summary_rows.append(
                {
                    'image_file_name': row['image_file_name'],
                    'sample_number': row.get('sample_number', ''),
                    'sample_id': row.get('sample_id', ''),
                    'label': row['label'],
                    'input_mode': input_mode,
                    'processed_output_file': str(output_path),
                    **metadata,
                }
            )
            if bool(metadata.get('segmentation_failed', False)):
                failure_rows.append({'image_file_name': row['image_file_name'], 'error': 'segmentation_failed'})
        except Exception as exc:
            failure_rows.append({'image_file_name': row.get('image_file_name', ''), 'error': str(exc)})

    return pd.DataFrame(records), pd.DataFrame(summary_rows), pd.DataFrame(failure_rows)

finish_notebook_cell_progress(NB_02_ROI_PREPROCESSING_CELL_PROGRESS_2)


[START] 02_roi_preprocessing.ipynb | Define preprocessing helpers [0/1 step] elapsed=0.0s
[RUNNING] 02_roi_preprocessing.ipynb | Define preprocessing helpers | done [0/1 step] elapsed=0.0s
[RUNNING] 02_roi_preprocessing.ipynb | Define preprocessing helpers | done [1/1 step] elapsed=0.0s


In [6]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_02_ROI_PREPROCESSING_CELL_PROGRESS_3 = start_notebook_cell_progress('02_roi_preprocessing.ipynb', 'Preprocess dataset', total_steps=1)

AUDITED_MANIFEST_PATH = Path(str(override('AUDITED_MANIFEST_PATH', GENERATED_SPLITS_ROOT / 'audited_manifest.csv')))
PROCESSED_MANIFEST_PATH = Path(str(override('PROCESSED_MANIFEST_PATH', GENERATED_SPLITS_ROOT / 'processed_manifest.csv')))
default_output_root = ROBOFLOW_PROCESSED_ROOT if DATASET_SOURCE == 'roboflow' else resolve_input_root(INPUT_MODE)
OUTPUT_ROOT = Path(str(override('OUTPUT_ROOT', default_output_root)))
PREPROCESSING_SUMMARY_PATH = Path(str(override('PREPROCESSING_SUMMARY_PATH', OUTPUT_ROOT / 'preprocessing_summary.csv')))
PREPROCESSING_FAILURES_PATH = Path(str(override('PREPROCESSING_FAILURES_PATH', OUTPUT_ROOT / 'preprocessing_failures.csv')))
BACKGROUND_MODE = str(override('BACKGROUND_MODE', 'gray'))
FORCE_REPROCESS = bool(override('FORCE_REPROCESS', False))

audited_df = pd.read_csv(AUDITED_MANIFEST_PATH, dtype=str).fillna('')
ensure_dir(PROCESSED_MANIFEST_PATH.parent)
ensure_dir(PREPROCESSING_SUMMARY_PATH.parent)
ensure_dir(PREPROCESSING_FAILURES_PATH.parent)

use_existing_processed = (
    not FORCE_REPROCESS
    and not audited_df.empty
    and set(audited_df['source_manifest_type'].astype(str).str.strip()) == {'processed_summary'}
)

if use_existing_processed and INPUT_MODE == 'processed_hsv_lab_threshold_roi_224':
    processed_df, summary_df, failures_df = normalize_existing_processed_manifest(audited_df)
else:
    processed_df, summary_df, failures_df = preprocess_audited_manifest(
        audited_df,
        output_root=OUTPUT_ROOT,
        input_mode=INPUT_MODE,
        background_mode=BACKGROUND_MODE,
    )

processed_df.to_csv(PROCESSED_MANIFEST_PATH, index=False)
summary_df.to_csv(PREPROCESSING_SUMMARY_PATH, index=False)
failures_df.to_csv(PREPROCESSING_FAILURES_PATH, index=False)

print(f'Processed rows: {len(processed_df)}')
print(f'Preprocessing failures: {len(failures_df)}')

finish_notebook_cell_progress(NB_02_ROI_PREPROCESSING_CELL_PROGRESS_3)


[START] 02_roi_preprocessing.ipynb | Preprocess dataset [0/1 step] elapsed=0.0s
[START] 02_roi_preprocessing.ipynb | preprocess processed_hsv_lab_threshold_roi_224 [0/3724 image] elapsed=0.0s
[RUNNING] 02_roi_preprocessing.ipynb | preprocess processed_hsv_lab_threshold_roi_224 [101/3724 image] elapsed=5.0s
[RUNNING] 02_roi_preprocessing.ipynb | preprocess processed_hsv_lab_threshold_roi_224 [201/3724 image] elapsed=10.0s
[RUNNING] 02_roi_preprocessing.ipynb | preprocess processed_hsv_lab_threshold_roi_224 [299/3724 image] elapsed=15.1s
[RUNNING] 02_roi_preprocessing.ipynb | preprocess processed_hsv_lab_threshold_roi_224 [372/3724 image] elapsed=18.8s
[RUNNING] 02_roi_preprocessing.ipynb | preprocess processed_hsv_lab_threshold_roi_224 [496/3724 image] elapsed=23.8s
[RUNNING] 02_roi_preprocessing.ipynb | preprocess processed_hsv_lab_threshold_roi_224 [609/3724 image] elapsed=28.9s
[RUNNING] 02_roi_preprocessing.ipynb | preprocess processed_hsv_lab_threshold_roi_224 [720/3724 image] elap

In [ ]:
from meatlens_pork_pipeline.notebook_progress import finish_notebook_cell_progress, start_notebook_cell_progress
NB_02_ROI_EMPTY_PROGRESS = start_notebook_cell_progress('02_roi_preprocessing.ipynb', 'Finalize preprocessing', total_steps=1)
finish_notebook_cell_progress(NB_02_ROI_EMPTY_PROGRESS)


In [ ]:
from meatlens_pork_pipeline.notebook_progress import finish_notebook_cell_progress, start_notebook_cell_progress
NB_02_ROI_EMPTY_PROGRESS = start_notebook_cell_progress('02_roi_preprocessing.ipynb', 'Finalize preprocessing', total_steps=1)
finish_notebook_cell_progress(NB_02_ROI_EMPTY_PROGRESS)
